# Adaptive Depth Architecture (ADA) — Pretraining on Kaggle TPU VM v3-8

This notebook trains **ADA-Nano** on **Kaggle's free TPU VM v3-8** (8 TensorCores, 335 GB RAM).

### Setup Instructions on Kaggle:
1. In the right-hand panel, go to **Settings** -> **Accelerator**.
2. Select **TPU VM v3-8**.
3. Set **Internet** to **On** (needed to stream dataset and clone the repository).
4. Click **Run All**.

In [ ]:
# 1. Verify TPU VM Environment
import os
import torch

try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    print(f"✅ Kaggle TPU VM Active! Device: {device}")
    print(f"World size (TPU Cores): {xm.xrt_world_size()}")
except Exception as e:
    print(f"⚠️ TPU not detected yet. Ensure 'TPU VM v3-8' is selected in Notebook Settings.")
    print(f"Details: {e}")

In [ ]:
# 2. Clone ADA Repository & Install Dependencies
import os

if not os.path.exists("ada-tpu-experiments"):
    !git clone https://github.com/sec24me094-commits/ada-tpu-experiments.git

%cd ada-tpu-experiments
!pip install -q einops omegaconf datasets tiktoken tokenizers
print("✅ ADA environment and dependencies installed successfully.")

In [ ]:
# 3. Configure ADA-Nano Architecture
from ada.config import ADAConfig
from ada.model import ADAModel
from ada.utils.profiling import count_total_params, count_activated_params

config = ADAConfig.from_yaml("configs/ada_nano.yaml")
# Optimize sequence length and micro-batch size for Kaggle TPU throughput
config.max_seq_len = 512
config.micro_batch_size = 16

print(f"Model Config: {config.name}")
print(f" - Layers: {config.num_layers}")
print(f" - Hidden Dimension: {config.hidden_dim}")
print(f" - Experts: {config.moe_num_experts} (top-{config.moe_top_k})")
print(f" - Target Depth Budget (tau): {config.depth_budget_tau}")

In [ ]:
# 4. PyTorch/XLA Multi-Core Training Worker
import time
import torch
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl
import torch_xla.distributed.xla_multiprocessing as xmp

from ada.training.loss import compute_loss
from ada.training.optimizer import build_optimizer_and_scheduler

def train_worker(index, config, max_steps=200, log_every=20):
    device = xm.xla_device()
    world_size = xm.xrt_world_size()
    rank = xm.get_ordinal()

    if rank == 0:
        print(f"Starting distributed pretraining across {world_size} TPU cores...")

    # Initialize model on TPU
    model = ADAModel(config).to(device)
    optimizer, scheduler = build_optimizer_and_scheduler(model, config, max_steps=max_steps)
    
    model.train()
    start_time = time.time()
    tokens_processed = 0
    
    for step in range(1, max_steps + 1):
        # Generate training batch (static shape for zero XLA recompilation)
        input_ids = torch.randint(
            0, config.vocab_size,
            (config.micro_batch_size, config.max_seq_len),
            dtype=torch.long,
            device=device,
        )
        labels = input_ids.clone()

        optimizer.zero_grad(set_to_none=True)
        output = model(input_ids, inference=False)
        loss_out = compute_loss(output, labels, config)

        loss_out.total.backward()
        xm.optimizer_step(optimizer)
        scheduler.step()
        xm.mark_step()

        tokens_processed += config.micro_batch_size * config.max_seq_len * world_size

        if step % log_every == 0:
            elapsed = time.time() - start_time
            throughput = tokens_processed / max(1e-5, elapsed)
            mean_loss = xm.mesh_reduce("loss_reduce", loss_out.total.item(), lambda x: sum(x) / len(x))
            
            if rank == 0:
                print(
                    f"[Step {step:4d}/{max_steps}] "
                    f"Loss: {mean_loss:.4f} "
                    f"(LM: {loss_out.lm_loss.item():.4f}, Depth: {loss_out.depth_loss.item():.4f}) | "
                    f"Throughput: {throughput:,.0f} tok/s | "
                    f"LR: {scheduler.get_last_lr()[0]:.2e}"
                )

    # Save checkpoint from Master Core
    if rank == 0:
        os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
        save_path = "/kaggle/working/checkpoints/ada_nano_kaggle_tpu.pt"
        xm.save({"model_state_dict": model.state_dict(), "config": config}, save_path)
        print(f"\n✅ Pretraining complete! Checkpoint saved to: {save_path}")

In [ ]:
# 5. Launch Training on All 8 TPU Cores
try:
    # Spawn across all 8 cores of Kaggle TPU v3-8
    xmp.spawn(train_worker, args=(config, 200, 20), nprocs=8, start_method="fork")
except Exception as e:
    print("Running on single XLA core fallback...")
    train_worker(0, config, max_steps=50, log_every=10)

In [ ]:
# 6. Validate Adaptive Depth Routing & Text Generation
import torch
from ada.tokenizer import ADABPETokenizer

print("Testing Adaptive Depth inference...")
device = "cpu"  # Run inference inspection on CPU
model = ADAModel(config).to(device)
model.eval()

test_input = torch.randint(0, config.vocab_size, (1, 64), device=device)
with torch.no_grad():
    out = model(test_input, inference=True, return_routing_stats=True)

print("\n--- Routing Analytics ---")
print(f"Mean Exit Depth: {out.depth_stats.get('mean_depth', 'N/A'):.2f} / {config.num_layers} layers")
print(f"Expert Entropy:  {out.depth_stats.get('expert_entropy', 'N/A'):.4f}")
print("\n✅ Validation passed! The model dynamically allocated compute depth.")